# PromoSense — Phase 6: Price Elasticity & Cannibalization Modeling (Google Colab)

This notebook provides a cloud-executable companion for the PromoSense Phase 6 workflow:
- **Own-Price Elasticity**: Interpretable Log-Log OLS, Promotional & Seasonal Controls, Base Shelf vs Discount Depth Decomposition, and Non-Linear LightGBM response curves.
- **Brand Tier Heterogeneity**: National Brand vs Private Label elasticity interaction modeling.
- **Cannibalization Analysis**: Joint cross-price elasticity estimation across direct substitute SKUs and counterfactual revenue-offset simulations.

## 1. Environment Setup & Dependencies
Install required dependencies in the Colab runtime.

In [ ]:
# Install project dependencies in Colab
!pip install -q lightgbm pandas numpy pyarrow scikit-learn scipy psutil joblib matplotlib

## 2. Mount Repository & Set Python Path
Ensure the PromoSense repository root is in `sys.path`.

In [ ]:
import os
import sys

PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print(f"Project root added to sys.path: {PROJECT_ROOT}")

## 3. Run Full Phase 6 Experiment Pipeline
Executes the end-to-end elasticity pipeline and logs resource usage.

In [ ]:
from ml_pipeline.src.elasticity.runner import Phase6ExperimentRunner

runner = Phase6ExperimentRunner()
results = runner.run()

print("\n--- Execution Completed Successfully ---")
print(f"Runtime: {results['metadata']['total_execution_time_seconds']}s")
print(f"Peak Memory: {results['metadata']['peak_memory_mb']} MB")

## 4. Inspect Own-Price Elasticities Across Specifications

In [ ]:
import pandas as pd

rows = []
for pid, data in results["own_price_elasticities"].items():
    models = data["models"]
    rows.append({
        "Product ID": pid,
        "Commodity": data["commodity"],
        "Brand Type": data["brand_type"],
        "Unadjusted OLS": models["unadjusted_paid_ols"]["elasticity"],
        "Controlled OLS": models["controlled_paid_ols"]["elasticity"],
        "Base Shelf OLS": models["decomposed_shelf_discount_ols"]["base_shelf_elasticity"],
        "Discount Depth Beta": models["decomposed_shelf_discount_ols"]["discount_elasticity"],
        "LightGBM Implied": models["lightgbm_monotonic"]["val_eval"]["model_implied_elasticity"],
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

## 5. Inspect Cannibalization & Counterfactual Revenue Offset

In [ ]:
cann_rows = []
for pair_name, data in results["cannibalization_analysis"].items():
    val_sim = data["val_simulation"]
    diffs = val_sim["counterfactual_differences"]
    metrics = val_sim["cannibalization_metrics"]
    cann_rows.append({
        "Pair Name": pair_name,
        "Own Elast A": data["elasticities"]["eps_AA_own"],
        "Cross Elast A->B": data["elasticities"]["eps_BA_cross"],
        "Delta Rev Promoted ($)": diffs["delta_revenue_A"],
        "Delta Rev Substitute ($)": diffs["delta_revenue_B"],
        "Revenue-Offset Ratio": metrics["revenue_offset_ratio"],
        "Net Category Lift ($)": diffs["net_category_revenue_lift"],
    })

display(pd.DataFrame(cann_rows))